# 🔵 Phase 2 — Hierarchical Model (No Optimization Yet)
##### Experiment 4 — Hierarchical Basic.
### Stage 1:
#### XGBoost (binary)

### Stage 2:
#### Random Forest (multiclass)

##### No heavy tuning yet.
##### Compare against Phase 1.

### This demonstrates the architectural benefit.

In [2]:
import sklearn
#Import necessary libraries
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from imblearn.under_sampling import RandomUnderSampler
import pandas as pd


data = pd.read_csv('UNSW_NB15.csv')
data = data.sample(n=250000, random_state=42)


# One hot encodin
# Assuming 'data' is your dataset containing both numerical and categorical columns
# Use the appropriate columns that contain categorical data
categorical_cols = ['proto', 'service','state']  # Replace with your categorical column names

# Perform one-hot encoding for categorical columns
data_encoded = pd.get_dummies(data, columns=categorical_cols)

# Split the encoded data into features and target
features_encoded = data_encoded.drop(['attack_cat', 'label'], axis=1)
# Replace 'target_column' with your target column name
target = data_encoded['attack_cat']  # Replace 'target_column' with your target column name
print(target.value_counts())
# Convert multi-class to binary
y_binary = (target != 'Normal').astype(int)

# Assuming you already have your preprocessed 'features_encoded' and 'target'
# Split the encoded data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(features_encoded, y_binary, test_size=0.2, random_state=42)


import numpy as np

neg = np.sum(y_train == 0)
pos = np.sum(y_train == 1)

scale_pos_weight = neg / pos
print("scale_pos_weight:", scale_pos_weight)

xgb_binary = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    scale_pos_weight=scale_pos_weight,
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

xgb_binary.fit(X_train, y_train)

from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

y_pred = xgb_binary.predict(X_test)
y_prob = xgb_binary.predict_proba(X_test)[:, 1]

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nROC-AUC:", roc_auc_score(y_test, y_prob))

attack_cat
Normal            90254
Generic           57073
Exploits          43175
Fuzzers           23580
DoS               15834
Reconnaissance    13580
Analysis           2607
Backdoor           2262
Shellcode          1465
Worms               170
Name: count, dtype: int64
scale_pos_weight: 0.5642597922662996
Confusion Matrix:
[[17994   116]
 [  531 31359]]

Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.99      0.98     18110
           1       1.00      0.98      0.99     31890

    accuracy                           0.99     50000
   macro avg       0.98      0.99      0.99     50000
weighted avg       0.99      0.99      0.99     50000


ROC-AUC: 0.999342926982402


In [ ]:
# Create the attack only training data
import sklearn
#Import necessary libraries
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from imblearn.under_sampling import RandomUnderSampler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.ensemble import RandomForestClassifier
import pandas as pd
import numpy as np


data = pd.read_csv('UNSW_NB15.csv')
data = data.sample(n=250000, random_state=42)


# One hot encodin
# Assuming 'data' is your dataset containing both numerical and categorical columns
# Use the appropriate columns that contain categorical data
categorical_cols = ['proto', 'service','state']  # Replace with your categorical column names

# Perform one-hot encoding for categorical columns
data_encoded = pd.get_dummies(data, columns=categorical_cols)

# Split the encoded data into features and target
features_encoded = data_encoded.drop(['attack_cat', 'label'], axis=1)

# Replace 'target_column' with your target column name
target = data_encoded['attack_cat']  # Replace 'target_column' with your target column name
print(target.value_counts())
# Convert multi-class to binary
# y_binary = (target != 'Normal').astype(int)
target_binary = data_encoded['label']

# Assuming you already have your preprocessed 'features_encoded' and 'target'
# Split the encoded data into training and testing sets
X_train, X_test, y_binary_train, y_binary_test, y_multi_train, y_multi_test = train_test_split(
    features_encoded,
    target_binary,
    target,              # original attack_cat
    test_size=0.2,
    stratify=target_binary,
    random_state=42
)


neg = np.sum(y_binary_train == 0)
pos = np.sum(y_binary_train == 1)

scale_pos_weight = neg / pos
print("scale_pos_weight:", scale_pos_weight)
xgb_binary = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    scale_pos_weight=scale_pos_weight,
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)
xgb_binary.fit(X_train, y_binary_train)

# Stage 1 predictions
y_stage1_pred = xgb_binary.predict(X_test)
y_prob = xgb_binary.predict_proba(X_test)[:, 1]

print("Confusion Matrix:")
print(confusion_matrix(y_binary_test, y_stage1_pred))

print("\nClassification Report:")
print(classification_report(y_binary_test, y_stage1_pred))

print("\nROC-AUC:", roc_auc_score(y_binary_test, y_prob))


# Keep only real attack samples
# Now only need the attack samples from the training set where target_binary == 1 
attack_train_idx = y_binary_train[y_binary_train == 1].index
print("These are the indexes where attacks rows are: ", attack_train_idx)
X_train_attack = X_train.loc[attack_train_idx]
y_train_attack = y_multi_train.loc[attack_train_idx]
print("Attacks without normal: ", y_train_attack)

# Now we perform undersampling and SMOTE-NC


rf_attack = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rf_attack.fit(X_train_attack, y_train_attack)

# Prepare final prediction array
attack_indices = np.where(y_stage1_pred == 1)[0]

final_predictions = np.array(['Normal'] * len(X_test))

if len(attack_indices) > 0:
    attack_preds = rf_attack.predict(X_test.iloc[attack_indices])
    final_predictions[attack_indices] = attack_preds


final_predictions = pd.Series(final_predictions)

from sklearn.metrics import classification_report, confusion_matrix

print(confusion_matrix(y_multi_test, final_predictions))
print(classification_report(y_multi_test, final_predictions))



attack_cat
Normal            90254
Generic           57073
Exploits          43175
Fuzzers           23580
DoS               15834
Reconnaissance    13580
Analysis           2607
Backdoor           2262
Shellcode          1465
Worms               170
Name: count, dtype: int64
scale_pos_weight: 0.5649819635828697
Confusion Matrix:
[[17948   103]
 [  554 31395]]

Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.99      0.98     18051
           1       1.00      0.98      0.99     31949

    accuracy                           0.99     50000
   macro avg       0.98      0.99      0.99     50000
weighted avg       0.99      0.99      0.99     50000


ROC-AUC: 0.9992670354691567
[[    0     0     0     0     0     0     0     0     0     0     0     0
      0     0     0     0     0]
 [   88     0    94     0   157   135     0    49     0     1     0    19
     13     0     0     0     0]
 [    0     0     0     0     0     0     0

C:\Users\HP\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\HP\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


                precision    recall  f1-score   support

        Analys       0.00      0.00      0.00         0
      Analysis       0.00      0.00      0.00       556
        Backdo       0.00      0.00      0.00         0
      Backdoor       0.00      0.00      0.00       440
           DoS       0.39      0.37      0.38      3083
        Exploi       0.00      0.00      0.00         0
      Exploits       0.00      0.00      0.00      8708
        Fuzzer       0.00      0.00      0.00         0
       Fuzzers       0.00      0.00      0.00      4648
        Generi       0.00      0.00      0.00         0
       Generic       0.00      0.00      0.00     11467
        Normal       0.97      0.99      0.98     18051
        Reconn       0.00      0.00      0.00         0
Reconnaissance       0.00      0.00      0.00      2703
        Shellc       0.00      0.00      0.00         0
     Shellcode       0.00      0.00      0.00       312
         Worms       0.67      0.31      0.43  

C:\Users\HP\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\HP\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\HP\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\metrics\_classification.py:183

In [ ]:
# Optuna Optimazation with resampling

# Create the attack only training data
import sklearn
#Import necessary libraries
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from imblearn.under_sampling import RandomUnderSampler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.ensemble import RandomForestClassifier
import pandas as pd
import numpy as np


data = pd.read_csv('UNSW_NB15.csv')
data = data.sample(n=250000, random_state=42)


# One hot encodin
# Assuming 'data' is your dataset containing both numerical and categorical columns
# Use the appropriate columns that contain categorical data
categorical_cols = ['proto', 'service','state']  # Replace with your categorical column names

# Perform one-hot encoding for categorical columns
data_encoded = pd.get_dummies(data, columns=categorical_cols)

# Split the encoded data into features and target
features_encoded = data_encoded.drop(['attack_cat', 'label'], axis=1)

# Replace 'target_column' with your target column name
target = data_encoded['attack_cat']  # Replace 'target_column' with your target column name
print(target.value_counts())
# Convert multi-class to binary
# y_binary = (target != 'Normal').astype(int)
target_binary = data_encoded['label']

# Assuming you already have your preprocessed 'features_encoded' and 'target'
# Split the encoded data into training and testing sets
X_train, X_test, y_binary_train, y_binary_test, y_multi_train, y_multi_test = train_test_split(
    features_encoded,
    target_binary,
    target,              # original attack_cat
    test_size=0.2,
    stratify=target_binary,
    random_state=42
)


neg = np.sum(y_binary_train == 0)
pos = np.sum(y_binary_train == 1)

scale_pos_weight = neg / pos
print("scale_pos_weight:", scale_pos_weight)
xgb_binary = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    scale_pos_weight=scale_pos_weight,
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)
xgb_binary.fit(X_train, y_binary_train)

# Stage 1 predictions
probs = xgb_binary.predict_proba(X_test)[:, 1]
y_stage1_pred = (probs > 0.3).astype(int)

print("Confusion Matrix:")
print(confusion_matrix(y_binary_test, y_stage1_pred))

print("\nClassification Report:")
print(classification_report(y_binary_test, y_stage1_pred))

print("\nROC-AUC:", roc_auc_score(y_binary_test, probs))


from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
import optuna
from imblearn.over_sampling import SMOTENC
import numpy as np

# Keep only real attack samples
# Now only need the attack samples from the training set where target_binary == 1 
attack_train_idx = y_binary_train[y_binary_train == 1].index
if len(attack_train_idx) == 0:
    raise ValueError("No attack samples found in training set")
print("These are the indexes where attacks rows are: ", attack_train_idx)
X_train_attack = X_train.loc[attack_train_idx]
y_train_attack = y_multi_train.loc[attack_train_idx]
print("Attacks without normal: ", y_train_attack)
cat_indices = [2, 3, 4]  # categorical column indices

def objective(trial):
    target_count = trial.suggest_int("target_count", 1000, 15000)
    n_estimators = trial.suggest_int("n_estimators", 100, 500)
    max_depth = trial.suggest_int("max_depth", 5, 30)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 10)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    fold_scores = []

    for train_idx, val_idx in skf.split(X_train_attack, y_train_attack):

        X_tr = X_train_attack.iloc[train_idx]
        y_tr = y_train_attack.iloc[train_idx]

        X_val_fold = X_train_attack.iloc[val_idx]
        y_val_fold = y_train_attack.iloc[val_idx]

        # Compute counts from THIS fold only
        counts = y_tr.value_counts()

        undersample_dict = {
            cls: min(count, target_count)
            for cls, count in counts.items()
        }

        # UnderSampling 
        rus = RandomUnderSampler(
            sampling_strategy=undersample_dict,
            random_state=42
        )

        X_res, y_res = rus.fit_resample(X_tr, y_tr)

        # SMOTE 
        smote = SMOTENC(
            categorical_features=cat_indices,
            sampling_strategy="not majority",
            random_state=42
        )

        X_res, y_res = smote.fit_resample(X_res, y_res)

        # RandomForest
        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            random_state=42,
            n_jobs=-1
        )

        model.fit(X_res, y_res)

        preds = model.predict(X_val_fold)

        fold_scores.append(
            f1_score(y_val_fold, preds, average="macro")
        )

    return np.mean(fold_scores)


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30)
best_trial = study.best_trial
print("Best target_count:", study.best_params)
print("Best F1:", study.best_value)
print("Best trial:", best_trial)

# Retrain using new best params
best_target_count = study.best_params["target_count"]
best_n_estimators = study.best_params["n_estimators"]
best_max_depth = study.best_params["max_depth"]
best_min_samples_split = study.best_params["min_samples_split"]

# Under-sample
counts = y_train_attack.value_counts()

undersample_dict = {
    cls: min(count, best_target_count)
    for cls, count in counts.items()
}

rus = RandomUnderSampler(
    sampling_strategy=undersample_dict,
    random_state=42
)

X_res, y_res = rus.fit_resample(X_train_attack, y_train_attack)

# SMOTE
cat_features = ["proto", "service", "state"]

cat_indices = [
    X_train_attack.columns.get_loc(col)
    for col in X_train_attack.columns
    if col in cat_features
]

smote = SMOTENC(
    categorical_features=cat_indices,
    sampling_strategy="not majority",
    random_state=42
)

X_res, y_res = smote.fit_resample(X_res, y_res)

rf_attack = RandomForestClassifier(
    n_estimators=best_n_estimators,
    max_depth=best_max_depth,
    min_samples_split=best_min_samples_split,
    random_state=42,
    n_jobs=-1
)

rf_attack.fit(X_res, y_res)

optuna.visualization.plot_optimization_history(study)
import optuna.visualization as vis

fig = vis.plot_optimization_history(study)
fig.show()

# Prepare final prediction array
attack_indices = np.where(y_stage1_pred == 1)[0]

final_predictions = np.array(['Normal'] * len(X_test))

if len(attack_indices) > 0:
    attack_preds = rf_attack.predict(X_test.iloc[attack_indices])
    final_predictions[attack_indices] = attack_preds


final_predictions = pd.Series(final_predictions)

from sklearn.metrics import classification_report, confusion_matrix

print(confusion_matrix(y_multi_test, final_predictions))
print(classification_report(y_multi_test, final_predictions))



attack_cat
Normal            90254
Generic           57073
Exploits          43175
Fuzzers           23580
DoS               15834
Reconnaissance    13580
Analysis           2607
Backdoor           2262
Shellcode          1465
Worms               170
Name: count, dtype: int64
scale_pos_weight: 0.5649819635828697


[I 2026-02-21 16:10:39,902] A new study created in memory with name: no-name-876c09fc-2e32-42af-9519-9698f626361a


Confusion Matrix:
[[17948   103]
 [  554 31395]]

Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.99      0.98     18051
           1       1.00      0.98      0.99     31949

    accuracy                           0.99     50000
   macro avg       0.98      0.99      0.99     50000
weighted avg       0.99      0.99      0.99     50000


ROC-AUC: 0.9992670354691567
These are the indexes where attacks rows are:  Index([148074, 168195,  53377, 191575, 135062, 181887, 187953, 156170,  94865,
        60977,
       ...
        87155, 187196, 138864, 184681, 193303, 112987, 156479, 167972, 150862,
       235203],
      dtype='int64', length=127797)
Attacks without normal:  148074     Generic
168195     Generic
53377     Exploits
191575     Generic
135062     Generic
            ...   
112987    Exploits
156479     Generic
167972         DoS
150862    Exploits
235203     Generic
Name: attack_cat, Length: 127797, dtype: str


[I 2026-02-21 16:11:00,357] Trial 0 finished with value: 0.7306092689679188 and parameters: {'target_count': 2994, 'n_estimators': 138, 'max_depth': 7, 'min_samples_split': 2}. Best is trial 0 with value: 0.7306092689679188.
[I 2026-02-21 16:14:12,155] Trial 1 finished with value: 0.7910373593292517 and parameters: {'target_count': 10689, 'n_estimators': 317, 'max_depth': 26, 'min_samples_split': 4}. Best is trial 1 with value: 0.7910373593292517.
[I 2026-02-21 16:16:32,226] Trial 2 finished with value: 0.7865285640410054 and parameters: {'target_count': 7561, 'n_estimators': 403, 'max_depth': 27, 'min_samples_split': 6}. Best is trial 1 with value: 0.7910373593292517.
[I 2026-02-21 16:19:10,816] Trial 3 finished with value: 0.787438840259927 and parameters: {'target_count': 8915, 'n_estimators': 392, 'max_depth': 30, 'min_samples_split': 4}. Best is trial 1 with value: 0.7910373593292517.
[I 2026-02-21 16:20:13,338] Trial 4 finished with value: 0.7760421790739079 and parameters: {'tar

Best target_count: {'target_count': 10689, 'n_estimators': 317, 'max_depth': 26, 'min_samples_split': 4}
Best Weighted F1: 0.7910373593292517


NameError: name 'rf_attack' is not defined